# Project 1 · Step 3 — DK1/DK2 prices and the CfD reference price

**Purpose.** Build the price inputs for the cash-flow model of Nordsøen Midt (NSM, DK1) and Hesselø (HES, DK2):

1. baseload day-ahead prices by month and year,
2. frequency of negative-price hours,
3. a **placeholder** AAP-weighted monthly reference price, using a wind proxy until the ERA5 yield (step 2) is ready.

**Inputs** (from `eds_price_download.py --start 2016 --end 2025 --wind`):
`data/prices_hourly_utc_2016_2025.csv`, `data/offshore_wind_hourly_utc_2016_2025.csv`.

**Outputs** (`data/outputs/`): `price_stats_monthly.csv`, `price_stats_yearly.csv`, `reference_price_monthly_<source>.csv`, `reference_price_yearly_<source>.csv`.

**Swapping in the real yield.** Set `YIELD_SOURCE = "era5"` and point `ERA5_AAP_FILES` at the hourly AAP files from step 2 (format in section 4). Nothing else changes.

### Reference-price definition used here
$RP_m = \dfrac{\sum_h AAP_h \cdot \bar p^{\,0}_h}{\sum_h AAP_h}$, where $\bar p^{\,0}_h$ is the mean of the hour's MTU prices **after** negative MTU prices are set to 0
(Appendix 3 §2.1 eq. (1) and §2.6.3; contract_assumptions B4). Before Oct 2025 the MTU is the hour, so this is just $\max(p_h, 0)$.

### Assumptions in this notebook
| # | Assumption | Status |
|---|---|---|
| P1 | NSM settles against DK1, Hesselø against DK2 | Asm (verify) — contract_assumptions B5 |
| P2 | Settlement months are Danish calendar months (local time); hourly data are UTC | Asm — the tender texts say "month" without a time zone |
| P3 | "Negative-price hour" = hour whose average MTU price is < 0. Since Oct 2025 the share of negative 15-min MTUs is reported separately | Asm (definition choice) |
| P4 | Hours missing from the price file are excluded, not filled; counts are shown | Asm |
| P5 | Wind proxy = Energinet metered offshore wind in the same price area, normalised to a capacity factor by a 60-day rolling maximum, × MinCap | Asm — placeholder only |
| P6 | Proxy is patched in negative-price hours (where existing farms may have curtailed) by interpolating across the block, because AAP counts curtailed output (contract_assumptions C1–C2) | Asm |
| P7 | DK2 proxy is mostly Baltic farms (Kriegers Flak, Rødsand); Hesselø is in the Kattegat. Anholt is DK1. The proxy's weather pattern differs from Hesselø's | Known limitation |

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---- paths ---------------------------------------------------------------
DATA_DIR = Path("data")
YEARS = "2016_2025"
PRICE_FILE = DATA_DIR / f"prices_hourly_utc_{YEARS}.csv"
WIND_FILE = DATA_DIR / f"offshore_wind_hourly_utc_{YEARS}.csv"
OUT_DIR = DATA_DIR / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- sites (progress notes, "Decisions so far") ---------------------------
SITES = {
    "NSM": {"name": "Nordsøen Midt", "area": "DK1", "mincap_mw": 1005},
    "HES": {"name": "Hesselø", "area": "DK2", "mincap_mw": 795},
}

# ---- yield source switch --------------------------------------------------
YIELD_SOURCE = "proxy"  # "proxy" until step 2 is done, then "era5"
ERA5_AAP_FILES = {
    "NSM": Path("../step2/data/aap_hourly_NSM.csv"),
    "HES": Path("../step2/data/aap_hourly_HES.csv"),
}

# ---- proxy settings (P5, P6) ----------------------------------------------
PROXY_CAP_WINDOW_DAYS = 60
PATCH_CURTAILED_NEG_HOURS = True

LOCAL_TZ = "Europe/Copenhagen"
COLORS = {"DK1": "#2a78d6", "DK2": "#eb6834"}  # validated categorical pair

plt.rcParams.update({
    "figure.dpi": 110, "figure.figsize": (9, 3.6),
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#b5b4ad", "axes.labelcolor": "#3d3d3a", "axes.titlesize": 11,
    "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.grid": True, "axes.axisbelow": True, "grid.color": "#e6e6e3", "grid.linewidth": 0.8,
    "xtick.color": "#5f5e5a", "ytick.color": "#5f5e5a",
    "lines.linewidth": 2, "legend.frameon": False,
})


def local_month(ts: pd.Series) -> pd.Series:
    # Danish calendar month of a UTC timestamp (P2).
    return ts.dt.tz_convert(LOCAL_TZ).dt.tz_localize(None).dt.to_period("M")

## 1. Load prices and check coverage

In [ ]:
prices = pd.read_csv(PRICE_FILE)
prices["time_utc"] = pd.to_datetime(prices["time_utc"], utc=True)
prices["month"] = local_month(prices["time_utc"])
prices["year"] = prices["month"].dt.year
prices["complete"] = prices["complete"].astype(bool)

# The UTC download window cuts the first/last Danish month (e.g. 2025-12-31 23:00 UTC is already
# January 2026 in Copenhagen). Drop an edge month if it has < 90 % of its calendar hours.
hrs = prices.groupby("month")["time_utc"].nunique()
full_hrs = pd.Series(hrs.index.days_in_month * 24, index=hrs.index)
drop = [m for m in (hrs.index.min(), hrs.index.max()) if hrs[m] < 0.9 * full_hrs[m]]
prices = prices[~prices["month"].isin(drop)]
if drop:
    print("Dropped partial edge months:", ", ".join(map(str, drop)))

cov = (prices.assign(missing=prices["price_dkk"].isna(),
                     partial=~prices["complete"] & prices["price_dkk"].notna())
             .groupby(["area", "year"])[["missing", "partial"]].sum())
print(f"Rows: {len(prices):,}  ·  {prices['time_utc'].min()} → {prices['time_utc'].max()}")
print(f"Missing hours (excluded, P4): {int(cov['missing'].sum())}  ·  partial hours (fewer MTUs than expected, kept): {int(cov['partial'].sum())}")
display(cov[(cov > 0).any(axis=1)] if (cov > 0).any(axis=None) else "No gaps.")
print("Sources by period:")
display(prices.groupby("source")["time_utc"].agg(["min", "max", "count"]))

## 2. Baseload prices and negative-price frequency (by month and year)

* `baseload_dkk` — time-weighted mean of hourly prices (negative prices included).
* `baseload_floored_dkk` — same with negatives floored at MTU level; the reference price can be compared with this.
* `neg_hours`, `neg_share` — hours with average price < 0 (P3). `zero_or_neg_hours` adds hours at exactly 0 or below.
* `hours_any_neg_mtu`, `neg_mtu_share` — only differ from the hourly measures since Oct 2025 (15-min MTU).

In [ ]:
p = prices.dropna(subset=["price_dkk"]).copy()
p["is_neg"] = p["price_dkk"] < 0
p["is_zero_or_neg"] = p["price_dkk"] <= 0
p["any_neg_mtu"] = p["frac_neg_mtu"] > 0


def price_stats(df: pd.DataFrame, by: str) -> pd.DataFrame:
    g = df.groupby(["area", by])
    out = g.agg(
        hours=("price_dkk", "size"),
        baseload_dkk=("price_dkk", "mean"),
        baseload_floored_dkk=("price_dkk_floored", "mean"),
        baseload_eur=("price_eur", "mean"),
        neg_hours=("is_neg", "sum"),
        zero_or_neg_hours=("is_zero_or_neg", "sum"),
        hours_any_neg_mtu=("any_neg_mtu", "sum"),
        neg_mtu_share=("frac_neg_mtu", "mean"),
    )
    out["neg_share"] = out["neg_hours"] / out["hours"]
    return out


stats_m = price_stats(p, "month")
stats_y = price_stats(p, "year")

yearly_view = stats_y[["baseload_dkk", "neg_hours", "neg_share"]].unstack("area")
display(yearly_view.style.format({c: "{:.0f}" for c in yearly_view.columns if c[0] != "neg_share"}
                                | {c: "{:.1%}" for c in yearly_view.columns if c[0] == "neg_share"}))

In [ ]:
fig, ax = plt.subplots()
for area in ["DK1", "DK2"]:
    s = stats_m.loc[area, "baseload_dkk"]
    ax.plot(s.index.to_timestamp(), s.values, color=COLORS[area], label=area)
ax.set_title("Monthly baseload day-ahead price")
ax.set_ylabel("DKK/MWh (nominal)")
ax.axhline(0, color="#b5b4ad", lw=0.8)
ax.legend(loc="upper left", ncols=2)
plt.show()

fig, ax = plt.subplots()
years = stats_y.index.get_level_values("year").unique()
w = 0.38
for i, area in enumerate(["DK1", "DK2"]):
    v = stats_y.loc[area, "neg_hours"].reindex(years)
    ax.bar(years + (i - 0.5) * (w + 0.02), v.values, width=w, color=COLORS[area], label=area)
ax.set_title("Hours with a negative day-ahead price (hourly average < 0)")
ax.set_ylabel("hours per year")
ax.set_xticks(years)
ax.grid(axis="x", visible=False)
ax.legend(loc="upper left", ncols=2)
plt.show()

## 3. Monthly profile of negative prices (pooled over all years)
Negative prices cluster in windy, low-demand months and around midday when solar is high, which is when an offshore farm produces too — the reason the AAP-weighted reference price sits below baseload.

In [ ]:
p["cal_month"] = p["month"].dt.month
prof = (p.groupby(["area", "cal_month"])["is_neg"].mean().unstack("area"))
display(prof.style.format("{:.1%}").set_caption("Share of hours with negative price, by calendar month"))

## 4. Hourly AAP series — proxy now, ERA5 later

**ERA5 yield file format (to be exported by `step2/yield_model.ipynb`):** one CSV per site,

| column | meaning |
|---|---|
| `time_utc` | hour-beginning, UTC, ISO 8601 (e.g. `2016-01-01 00:00:00+00:00`) |
| `aap_mwh` | potential output of the **MinCap** turbines in that hour: net of internal and external wakes, × (1 − unscheduled unavailability), ≤ MinCap (contract_assumptions B1–B3, B12, C3) |
| `gen_mwh` *(optional)* | metered output after electrical losses; used in step 5, not here |

⚠ ERA5 wind fields are instantaneous values at the top of each hour. To get an hour-beginning energy, average the power at *t* and *t*+1 h (or state the convention used). A one-hour shift moves the reference price only slightly, but it should be consistent.

Keep ERA5 and prices on the **same historical hours** (2016–2025 overlap). Pairing a wind year with a different price year would break the wind–price correlation that the reference price depends on.

**Why the proxy is good enough for now.** Within a month the reference price only depends on the *shape* of AAP, not its level, so the proxy's unknown installed capacity cancels out. The 60-day rolling maximum only matters for weighting months against each other in the yearly figure.

In [ ]:
def proxy_aap(area: str, mincap_mw: float, px: pd.DataFrame) -> pd.Series:
    # Placeholder AAP (MWh per hour) from Energinet metered offshore wind in `area` (P5–P7).
    w = pd.read_csv(WIND_FILE)
    w = w[w["area"] == area].copy()
    w["time_utc"] = pd.to_datetime(w["time_utc"], utc=True)
    gen = w.set_index("time_utc")["offshore_mwh"].astype(float).sort_index()

    if PATCH_CURTAILED_NEG_HOURS:
        neg = px.set_index("time_utc")["price_dkk"].reindex(gen.index) < 0
        filled = gen.mask(neg).interpolate(method="time", limit_area="inside")
        gen = gen.where(~neg, np.fmax(gen, filled))

    cap = gen.rolling(f"{PROXY_CAP_WINDOW_DAYS}D", center=True, min_periods=24 * 7).max()
    cf = (gen / cap).clip(0, 1)
    return (cf * mincap_mw).rename("aap_mwh")


def era5_aap(site: str, mincap_mw: float) -> pd.Series:
    f = ERA5_AAP_FILES[site]
    if not f.exists():
        raise FileNotFoundError(f"{f} not found — run step 2 and export hourly AAP first.")
    df = pd.read_csv(f)
    missing = {"time_utc", "aap_mwh"} - set(df.columns)
    if missing:
        raise ValueError(f"{f} lacks columns {missing}")
    s = df.set_index(pd.to_datetime(df["time_utc"], utc=True))["aap_mwh"].astype(float).sort_index()
    assert s.index.is_unique, "duplicate timestamps in yield file"
    steps = s.index.to_series().diff().dropna().value_counts()
    assert steps.index[0] == pd.Timedelta(hours=1), f"yield file is not hourly: {steps.head()}"
    if (s > mincap_mw * 1.001).any():
        raise ValueError("aap_mwh exceeds MinCap; AAP is capped at Minimum Capacity (B12)")
    return s.rename("aap_mwh")


def load_aap(site: str) -> pd.Series:
    cfg = SITES[site]
    if YIELD_SOURCE == "era5":
        return era5_aap(site, cfg["mincap_mw"])
    px = prices[prices["area"] == cfg["area"]]
    return proxy_aap(cfg["area"], cfg["mincap_mw"], px)


aap = {site: load_aap(site) for site in SITES}
for site, s in aap.items():
    cf = s.mean() / SITES[site]["mincap_mw"]
    print(f"{site} ({YIELD_SOURCE}): {s.notna().sum():,} hours, mean CF {cf:.1%}")
if YIELD_SOURCE == "proxy":
    print("\nProxy CFs are not site yields (they reflect the existing fleet); only the hourly shape is used.")

## 5. Reference price (placeholder)

Per Danish calendar month:

* `rp_dkk` — contract definition: AAP-weighted, negatives floored at MTU level.
* `rp_hourfloor_dkk` — same, but floored after hourly averaging. Equal to `rp_dkk` before Oct 2025; the gap after shows the effect of the 15-min MTU.
* `capture_dkk` — AAP-weighted price **without** the floor (what an unhedged merchant farm that never curtails would see).
* `capture_factor` — `rp_dkk / baseload_dkk`.
* `aap_share_neg` — share of the month's AAP in negative-price hours: volume that is still settled under the CfD but earns nothing on the market (C1).
* `coverage` — share of the month's hours with both AAP and price; months below 95 % are flagged.

In [ ]:
def reference_price(site: str) -> pd.DataFrame:
    area = SITES[site]["area"]
    px = prices[prices["area"] == area].set_index("time_utc")
    df = px[["price_dkk", "price_dkk_floored", "month"]].join(aap[site], how="left")
    n_hours = df.groupby("month").size()
    df = df.dropna(subset=["price_dkk", "aap_mwh"])

    df["rev_floor"] = df["aap_mwh"] * df["price_dkk_floored"]
    df["rev_hourfloor"] = df["aap_mwh"] * df["price_dkk"].clip(lower=0)
    df["rev_raw"] = df["aap_mwh"] * df["price_dkk"]
    df["aap_neg"] = df["aap_mwh"].where(df["price_dkk"] < 0, 0.0)

    m = df.groupby("month")[["aap_mwh", "rev_floor", "rev_hourfloor", "rev_raw", "aap_neg"]].sum()
    m["hours_used"] = df.groupby("month").size()
    m["coverage"] = m["hours_used"] / n_hours.reindex(m.index)
    m["rp_dkk"] = m["rev_floor"] / m["aap_mwh"]
    m["rp_hourfloor_dkk"] = m["rev_hourfloor"] / m["aap_mwh"]
    m["capture_dkk"] = m["rev_raw"] / m["aap_mwh"]
    m["aap_share_neg"] = m["aap_neg"] / m["aap_mwh"]
    m = m.join(stats_m.loc[area, ["baseload_dkk", "baseload_floored_dkk"]])
    m["capture_factor"] = m["rp_dkk"] / m["baseload_dkk"]
    m.insert(0, "area", area)
    m.insert(0, "site", site)
    return m


def to_yearly(m: pd.DataFrame) -> pd.DataFrame:
    # Yearly RP = AAP-weighted over the months (= sum of revenue / sum of AAP).
    g = m.groupby(m.index.year)
    y = g[["aap_mwh", "rev_floor", "rev_hourfloor", "rev_raw", "aap_neg"]].sum()
    y["rp_dkk"] = y["rev_floor"] / y["aap_mwh"]
    y["rp_hourfloor_dkk"] = y["rev_hourfloor"] / y["aap_mwh"]
    y["capture_dkk"] = y["rev_raw"] / y["aap_mwh"]
    y["aap_share_neg"] = y["aap_neg"] / y["aap_mwh"]
    area = m["area"].iloc[0]
    y["baseload_dkk"] = stats_y.loc[area, "baseload_dkk"].reindex(y.index)
    y["capture_factor"] = y["rp_dkk"] / y["baseload_dkk"]
    y.index.name = "year"
    y.insert(0, "area", area)
    y.insert(0, "site", m["site"].iloc[0])
    return y


rp_m = {s: reference_price(s) for s in SITES}
rp_y = {s: to_yearly(rp_m[s]) for s in SITES}

for s in SITES:
    low = rp_m[s][rp_m[s]["coverage"] < 0.95]
    if len(low):
        print(f"{s}: {len(low)} months with < 95 % coverage: {', '.join(map(str, low.index[:8]))}{' …' if len(low) > 8 else ''}")

cols = ["baseload_dkk", "rp_dkk", "capture_dkk", "capture_factor", "aap_share_neg"]
view = pd.concat({s: rp_y[s][cols] for s in SITES}, axis=1)
display(view.style.format("{:.0f}").format("{:.2f}", subset=[(s, "capture_factor") for s in SITES])
            .format("{:.1%}", subset=[(s, "aap_share_neg") for s in SITES])
            .set_caption(f"Yearly reference price (yield source: {YIELD_SOURCE})"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for ax, site in zip(axes, SITES):
    m = rp_m[site]
    t = m.index.to_timestamp()
    area = SITES[site]["area"]
    ax.plot(t, m["baseload_dkk"], color="#9a9991", lw=1.5, label="baseload")
    ax.plot(t, m["rp_dkk"], color=COLORS[area], label="reference price (AAP-weighted)")
    ax.set_title(f"{SITES[site]['name']} ({area})")
    ax.axhline(0, color="#b5b4ad", lw=0.8)
axes[0].set_ylabel("DKK/MWh (nominal)")
axes[0].legend(loc="upper left")
fig.suptitle(f"Monthly reference price vs baseload — yield source: {YIELD_SOURCE}", x=0.01, ha="left", fontsize=11)
fig.tight_layout()
plt.show()

fig, ax = plt.subplots()
for site in SITES:
    y = rp_y[site]
    area = SITES[site]["area"]
    ax.plot(y.index, y["capture_factor"], color=COLORS[area], marker="o", ms=5, label=f"{site} ({area})")
ax.set_title("Reference price as a share of baseload (capture factor)")
ax.set_ylabel("RP / baseload")
ax.legend(loc="lower left", ncols=2)
plt.show()

post = {s: rp_m[s].loc[rp_m[s].index >= pd.Period("2025-10", "M"), ["rp_dkk", "rp_hourfloor_dkk"]] for s in SITES}
for s, d in post.items():
    if len(d):
        print(f"{s}: MTU-level vs hour-level floor since Oct 2025, mean gap "
              f"{(d['rp_dkk'] - d['rp_hourfloor_dkk']).mean():+.2f} DKK/MWh over {len(d)} months")

## 6. Export for the cash-flow model (step 5)

In [ ]:
stats_m.reset_index().assign(month=lambda d: d["month"].astype(str)).to_csv(OUT_DIR / "price_stats_monthly.csv", index=False)
stats_y.reset_index().to_csv(OUT_DIR / "price_stats_yearly.csv", index=False)

keep = ["site", "area", "aap_mwh", "hours_used", "coverage", "baseload_dkk", "baseload_floored_dkk",
        "rp_dkk", "rp_hourfloor_dkk", "capture_dkk", "capture_factor", "aap_share_neg"]
rpm = pd.concat([rp_m[s][keep] for s in SITES]).reset_index().assign(month=lambda d: d["month"].astype(str))
rpm.to_csv(OUT_DIR / f"reference_price_monthly_{YIELD_SOURCE}.csv", index=False)
rpy = pd.concat([rp_y[s] for s in SITES]).reset_index()
rpy.to_csv(OUT_DIR / f"reference_price_yearly_{YIELD_SOURCE}.csv", index=False)
print("Saved:", *sorted(x.name for x in OUT_DIR.glob("*.csv")), sep="\n  ")

## Notes for later steps
* **Hedge identity (step 5).** If the farm curtails in negative-price MTUs (C1) and metered output equals AAP, merchant revenue is $\sum_h AAP_h\,\bar p^{\,0}_h = RP\cdot AAP$, so revenue collapses to $K \cdot AAP$. What is left exposed to prices is: the gap between metered output and AAP (electrical losses, unscheduled downtime), balancing costs, the net cap on state payments (B9), overplanted MW, and the post-CfD merchant tail. That is why the 2033–2052 price scenarios matter mainly for the net-cap test and the tail (see `price_scenarios_method.md`).
* All prices here are **nominal** DKK. Deflate with the NPI before comparing with the strike prices (2026 prices) or DEA projections (2025 prices).
* When the ERA5 yield arrives, re-run with `YIELD_SOURCE = "era5"` and compare the capture factors with the proxy run; large differences in DK2 would point to the proxy-geography issue (P7).